## Establishing the Architectural Baseline: 1-Dimensional Convolutional Neural Network (1D CNN)
A one-dimensional convolutional neural network (1D CNN) is a feedforward deep learning architecture
that applies learnable convolutional filters along a single spatial or temporal axis. Given an input
tensor $\mathbf{X} \in \mathbb{R}^{N \times C \times L}$, where $N$ is the batch size, $C$ is the
number of input channels, and $L$ is the sequence length, each convolutional layer computes a
feature map via the discrete cross-correlation:

$$y[i] = \sum_{k=0}^{K-1} w[k] \cdot x[i+k] + b$$

where $\mathbf{w} \in \mathbb{R}^{K}$ is the learnable kernel of size $K$, $b$ is a scalar bias,
and $i$ indexes the temporal dimension. Multiple filters extract parallel feature representations,
and successive layers capture hierarchical temporal abstractions of increasing receptive field.

### WHY 1D CNN for Time-Series Sensor Data?

Imagine your data as a **3D volume**:

- **X-axis:** Time frames (forward progression of the fall)
- **Y-axis:** Window size (50 frames = 1 second)
- **Z-axis:** Sensor features (Accel X, Accel Y, Gyro Z, etc.)

What the 1D CNN Does

The kernel moves **only along the time axis (X-axis)** while covering **all features (Z-axis)** at once.

- Kernel shape: `(features, kernel_size)` = `(10, 7)`
- Sweeps from `t=0` to `t=50` step by step

| Axis | Role | Kernel Movement |
|------|------|----------------|
| X (Time) | Sequential order | ✅ Slides along |
| Y (Window) | Fixed segment length | ❌ Does not move |
| Z (Features) | Multiple sensors | ✅ Fully covered |

## The Key Insight

> The kernel finds **temporal patterns** across **all sensors simultaneously** — like detecting "spike in Accel Y + drop in Gyro X" happening at the same moment in time.

No sensor mixing. No time destruction. Just pure temporal feature learning.

This cell implements the core MLOps preprocessing pipeline to transform raw CSV streams into PyTorch-ready 3D tensors while solving two major machine learning pitfalls:

**Phase 1: Leak-Proof Stratified Routing**

- **The Action:** Splits the dataset into Training (80%) and Validation (20%).
- **The Physics:** It strictly splits the dataset at the file level, rather than shuffling individual rows. This physically prevents temporal data leakage (where adjacent milliseconds of the same fall bleed into both train and test sets). Stratification ensures the exact ratio of Fall-to-Normal files is preserved.

**Phase 2: Dynamic Window Extraction**

- **The Action:** Sweeps across the CSVs, slicing the continuous sensor data into discrete 50-frame (1.0 second) chunks.
- **The Physics:** To solve the massive class imbalance (where 90% of data is just normal walking), it implements a Dynamic Stride. Normal walking windows step forward by 25 frames (standard 50% overlap). When a fall is detected, it triggers "burst mode," stepping forward by only 5 frames. This synthetically multiplies the minority fall data through heavy overlapping without needing to generate fake data.

> **Note:** Customized helper functions are implemented from this file onwards.  
> Readers can check out the [Utilities](utilities) folder or the [README](utilities/README.md) file to gain more intuition on the working of:  
> - `utilities.dataset_router`  
> - `utilities.window_extractor`  
> - `utilities.experiment_trainer`  
> - `utilities.experiment_tester`

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader


# Import custom MLOps Framework from the utilities folder
from utilities.dataset_router import DatasetRouter
from utilities.window_extractor import WindowExtractor
from utilities.experiment_trainer import ExperimentTrainer
from utilities.experiment_tester import ExperimentTester

print("=== 1. DATA ROUTING (Leak-Proof) ===")
TRAIN_DIR = "./DataSet/Sample_Training"
TEST_DIR = "./DataSet/Sample_Test"

router = DatasetRouter(dataset_path=TRAIN_DIR, test_size=0.20, random_state=42)
train_files, val_files = router.create_splits()

test_router = DatasetRouter(dataset_path=TEST_DIR, test_size=0.01) # Hack to load 100% as test
test_files, _ = test_router.create_splits()


print("\n=== 2. WINDOW EXTRACTION ===")
extractor = WindowExtractor(window_size=50, fall_threshold=0.4)

print("Extracting Train (Dynamic)...")
X_train_raw, y_train, _ = extractor.extract_dynamic(train_files, normal_stride=25, fall_stride=5)

print("Extracting Validation (Standard)...")
X_val_raw, y_val, _ = extractor.extract_standard(val_files, stride=25)

print("Extracting Test (Strict Overlap)...")
X_test_raw, y_test, test_log = extractor.extract_strict_overlap(test_files, overlap_fraction=0.5)


print("\n=== 3. FEATURE SCALING ===")
scaler = StandardScaler()
n_tr, t_st, n_f = X_train_raw.shape
X_train_scaled = scaler.fit_transform(X_train_raw.reshape(-1, n_f)).reshape(n_tr, t_st, n_f)

n_v = X_val_raw.shape[0]
X_val_scaled = scaler.transform(X_val_raw.reshape(-1, n_f)).reshape(n_v, t_st, n_f)

n_ts = X_test_raw.shape[0]
X_test_scaled = scaler.transform(X_test_raw.reshape(-1, n_f)).reshape(n_ts, t_st, n_f)


print("\n=== 4. PYTORCH DATASET SETUP ===")
class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        # Transpose transforms (SeqLen, Channels) to (Channels, SeqLen) for 1D CNNs
        return torch.tensor(self.X[idx].T, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


print("\n=== 5. BASELINE ARCHITECTURE ===")
class Baseline1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(10, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(128 * 25, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x)))) 
        x = F.relu(self.bn2(self.conv2(x)))            
        x = torch.flatten(x, 1)                        
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)


print("\n=== 6. EXECUTION ENGINE ===")
baseline_model = Baseline1DCNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr=0.001) 

trainer = ExperimentTrainer(
    exp_name="Exp_002_Baseline_1DCNN",
    description="Re-running baseline 1D CNN with strict MLOps extraction pipeline.",
    model=baseline_model,
    criterion=criterion,
    optimizer=optimizer
)

# Train the Baseline
best_model, metrics = trainer.train(
    X_train_scaled, y_train, 
    X_val_scaled, y_val, 
    epochs=30, 
    batch_size=256
)

# Run the final Blind Test
tester = ExperimentTester(
    exp_name="Exp_002_Baseline_1DCNN", 
    model_architecture=Baseline1DCNN()
)

preds, targets = tester.run_blind_test(
    X_test=X_test_scaled, 
    y_test=y_test, 
    test_name="Baseline CNN Blind Test", 
    extraction_method="Strict Overlap (50%)",
    batch_size=256
)

=== 1. DATA ROUTING (Leak-Proof) ===

--- Initializing File Router on DataSet/Sample_Training ---
Found 2473 total CSV files. Tagging...


Routing Files: 100%|██████████| 2473/2473 [00:34<00:00, 71.08it/s] 



=== ROUTING SUCCESSFUL ===
Train Files: 1978 (Falls: 1489)
Val Files:   495 (Falls: 373)

--- Initializing File Router on DataSet/Sample_Test ---
Found 586 total CSV files. Tagging...


Routing Files: 100%|██████████| 586/586 [00:06<00:00, 97.15it/s] 



=== ROUTING SUCCESSFUL ===
Train Files: 580 (Falls: 435)
Val Files:   6 (Falls: 4)

=== 2. WINDOW EXTRACTION ===
Extracting Train (Dynamic)...


...

Extracting Validation (Standard)...


Extracting (Standard): 100%|██████████| 495/495 [00:07<00:00, 62.04it/s]


Extracting Test (Strict Overlap)...


Extracting (50% Overlap): 100%|██████████| 580/580 [00:07<00:00, 73.04it/s]



=== 3. FEATURE SCALING ===

=== 4. PYTORCH DATASET SETUP ===

=== 5. BASELINE ARCHITECTURE ===

=== 6. EXECUTION ENGINE ===

=== LAUNCHING EXPERIMENT: Exp_002_Baseline_1DCNN ===
Device: cuda | Epochs: 30 | Batch: 256
Epoch [01/30] | Train Loss: 0.0989 | Val Loss: 0.0666 | Val F1: 0.8684 | Recall: 0.9248
Epoch [02/30] | Train Loss: 0.0631 | Val Loss: 0.0529 | Val F1: 0.8928 | Recall: 0.9248
Epoch [03/30] | Train Loss: 0.0530 | Val Loss: 0.0485 | Val F1: 0.8977 | Recall: 0.9163
Epoch [04/30] | Train Loss: 0.0495 | Val Loss: 0.0463 | Val F1: 0.8986 | Recall: 0.9090
Epoch [05/30] | Train Loss: 0.0470 | Val Loss: 0.0423 | Val F1: 0.9104 | Recall: 0.9126
Epoch [06/30] | Train Loss: 0.0420 | Val Loss: 0.0458 | Val F1: 0.9061 | Recall: 0.9248
Epoch [07/30] | Train Loss: 0.0392 | Val Loss: 0.0469 | Val F1: 0.9033 | Recall: 0.9235
Epoch [08/30] | Train Loss: 0.0378 | Val Loss: 0.0424 | Val F1: 0.9103 | Recall: 0.9175
Epoch [09/30] | Train Loss: 0.0348 | Val Loss: 0.0562 | Val F1: 0.8884 | Recal

tuning 1d cnn

In [6]:
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

print("=== INITIATING OPTUNA TUNING FOR 1D CNN ===")

# ==========================================
# 1. THE TUNABLE ARCHITECTURE
# ==========================================
class Tunable1DCNN(nn.Module):
    def __init__(self, filter_1, filter_2, kernel_size, dropout_rate):
        super().__init__()
        
        # Calculate padding to keep dimensions clean (prevents shape crashes)
        pad = kernel_size // 2 
        
        self.conv1 = nn.Conv1d(10, filter_1, kernel_size=kernel_size, padding=pad)
        self.bn1 = nn.BatchNorm1d(filter_1)
        self.pool = nn.MaxPool1d(2) 
        
        self.conv2 = nn.Conv1d(filter_1, filter_2, kernel_size=kernel_size, padding=pad)
        self.bn2 = nn.BatchNorm1d(filter_2)
        
        self.dropout = nn.Dropout(dropout_rate)
        
        # 50 frames cut in half by MaxPool = 25 frames.
        self.fc1 = nn.Linear(filter_2 * 25, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = torch.flatten(x, 1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)


# ==========================================
# 2. THE OPTUNA ENGINE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def cnn_objective(trial):
    # Determine the mutation geometry
    filter_1 = trial.suggest_categorical("filter_1", [32, 64, 128])
    filter_2 = trial.suggest_categorical("filter_2", [64, 128, 256])
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7]) # Time-receptive field
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    # Initialize model
    model = Tunable1DCNN(filter_1, filter_2, kernel_size, dropout_rate).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Prepare Data
    train_loader = DataLoader(FallDataset(X_train_scaled, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(FallDataset(X_val_scaled, y_val), batch_size=batch_size, shuffle=False)

    # Fast Training Loop (15 epochs)
    epochs = 15 
    best_val_f1 = 0.0

    for epoch in range(epochs):
        model.train()
        for b_X, b_y in train_loader:
            b_X, b_y = b_X.to(device), b_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(b_X), b_y)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for v_X, v_y in val_loader:
                v_X, v_y = v_X.to(device), v_y.to(device)
                logits = model(v_X)
                preds = (torch.sigmoid(logits) > 0.5).float()
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(v_y.cpu().numpy())

        # Metric calculation
        val_f1 = f1_score(all_targets, all_preds, zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            
        # Median Pruner
        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_f1

# ==========================================
# 3. EXECUTE SEARCH
# ==========================================
study_cnn = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study_cnn.optimize(cnn_objective, n_trials=30)

print("\n=== OPTIMIZATION COMPLETE ===")
print("Best Trial:")
cnn_trial = study_cnn.best_trial
print(f"  Best F1 Score: {cnn_trial.value:.4f}")
print("  Winning Hyperparameters: ")
for key, value in cnn_trial.params.items():
    print(f"    {key}: {value}")

/mnt/s/backup/DM_PROJECT/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-04 08:26:22,665] A new study created in memory with name: no-name-633ca1cb-f4bd-49d3-af2d-eb27dace8913


=== INITIATING OPTUNA TUNING FOR 1D CNN ===


[I 2026-04-04 08:27:14,568] Trial 0 finished with value: 0.9082125603864735 and parameters: {'filter_1': 32, 'filter_2': 64, 'kernel_size': 5, 'dropout_rate': 0.112592524146163, 'lr': 0.0001222798341302779, 'batch_size': 256}. Best is trial 0 with value: 0.9082125603864735.
[I 2026-04-04 08:28:36,194] Trial 1 finished with value: 0.9150326797385621 and parameters: {'filter_1': 64, 'filter_2': 256, 'kernel_size': 5, 'dropout_rate': 0.11820013770313197, 'lr': 0.0032437114042345627, 'batch_size': 128}. Best is trial 1 with value: 0.9150326797385621.
[I 2026-04-04 08:29:20,530] Trial 2 finished with value: 0.9113001215066828 and parameters: {'filter_1': 64, 'filter_2': 64, 'kernel_size': 7, 'dropout_rate': 0.18798252130652393, 'lr': 0.0027762168927116236, 'batch_size': 256}. Best is trial 1 with value: 0.9150326797385621.
[I 2026-04-04 08:30:27,270] Trial 3 finished with value: 0.9134906231094979 and parameters: {'filter_1': 128, 'filter_2': 64, 'kernel_size': 5, 'dropout_rate': 0.40087424


=== OPTIMIZATION COMPLETE ===
Best Trial:
  Best F1 Score: 0.9166
  Winning Hyperparameters: 
    filter_1: 32
    filter_2: 256
    kernel_size: 7
    dropout_rate: 0.2445619461375933
    lr: 0.00091529872740991
    batch_size: 64


In [ ]:
# ==========================================
# 1. THE FINAL HARDCODED BASELINE
# ==========================================
class FinalBaseline1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Optuna's Winning Specs
        filter_1 = 32
        filter_2 = 256
        kernel_size = 7
        self.dropout_rate = 0.2445
        
        pad = kernel_size // 2 
        
        self.conv1 = nn.Conv1d(10, filter_1, kernel_size=kernel_size, padding=pad)
        self.bn1 = nn.BatchNorm1d(filter_1)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(filter_1, filter_2, kernel_size=kernel_size, padding=pad)
        self.bn2 = nn.BatchNorm1d(filter_2)
        
        
        self.dropout = nn.Dropout(self.dropout_rate)
        
        self.fc1 = nn.Linear(filter_2 * 25, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = torch.flatten(x, 1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

# ==========================================
# 2. LAUNCH ULTIMATE BASELINE RUN
# ==========================================
print("=== LAUNCHING ULTIMATE BASELINE RUN ===")

final_cnn = FinalBaseline1DCNN()
criterion = nn.BCEWithLogitsLoss()
# Optuna's exact winning learning rate
optimizer = optim.Adam(final_cnn.parameters(), lr=0.000915) 

trainer = ExperimentTrainer(
    exp_name="Exp_002_Ultimate_1DCNN",
    description="Final run using Optuna parameters (f1=32, f2=256, k=7, dropout=0.24).",
    model=final_cnn,
    criterion=criterion,
    optimizer=optimizer
)

# Train for 40 epochs
best_model, metrics = trainer.train(
    X_train_scaled, y_train, 
    X_val_scaled, y_val, 
    epochs=40, 
    batch_size=64 # Optuna's winning batch size
)

# ==========================================
# 3. THE BLIND TEST
# ==========================================
tester = ExperimentTester(
    exp_name="Exp_002_Ultimate_1DCNN", 
    model_architecture=FinalBaseline1DCNN()
)

preds, targets = tester.run_blind_test(
    X_test=X_test_scaled, 
    y_test=y_test, 
    test_name="Ultimate CNN Blind Test", 
    extraction_method="Strict Overlap (50%)",
    batch_size=64
)

=== LAUNCHING ULTIMATE BASELINE RUN ===

=== LAUNCHING EXPERIMENT: Exp_002_Ultimate_1DCNN ===
Device: cuda | Epochs: 40 | Batch: 64
Epoch [01/40] | Train Loss: 0.0999 | Val Loss: 0.0586 | Val F1: 0.8762 | Recall: 0.9150
Epoch [02/40] | Train Loss: 0.0731 | Val Loss: 0.0516 | Val F1: 0.8878 | Recall: 0.9126
Epoch [03/40] | Train Loss: 0.0626 | Val Loss: 0.0530 | Val F1: 0.8981 | Recall: 0.9357
Epoch [04/40] | Train Loss: 0.0576 | Val Loss: 0.0449 | Val F1: 0.9030 | Recall: 0.9090
Epoch [05/40] | Train Loss: 0.0525 | Val Loss: 0.0497 | Val F1: 0.9083 | Recall: 0.9078
Epoch [06/40] | Train Loss: 0.0504 | Val Loss: 0.0506 | Val F1: 0.9038 | Recall: 0.9345
Epoch [07/40] | Train Loss: 0.0464 | Val Loss: 0.0467 | Val F1: 0.9140 | Recall: 0.9417
Epoch [08/40] | Train Loss: 0.0450 | Val Loss: 0.0475 | Val F1: 0.9072 | Recall: 0.9199
Epoch [09/40] | Train Loss: 0.0436 | Val Loss: 0.0489 | Val F1: 0.9021 | Recall: 0.9053
Epoch [10/40] | Train Loss: 0.0405 | Val Loss: 0.0507 | Val F1: 0.9062 | Rec

### Binary Focal Loss: Handling Class Imbalance

#### The Problem with Standard BCE

- Easy negatives (walking, resting) dominate the loss gradient
- Model optimizes for mundane activity instead of learning fall physics
- Result: False positives (heavy sit-down classified as fall)

#### The Solution: Focal Loss

**Origin:** Lin et al., FAIR (2017) for dense object detection

**Formula:**
$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

| Parameter | Value | Purpose |
|-----------|-------|---------|
| $\alpha$ (Alpha) | 0.85 | Heavily weights positive class (falls) |
| $\gamma$ (Gamma) | 2.0 | Down-weights easy examples |

### How It Works

| Scenario | Model Confidence | Modulating Factor $(1-p_t)^\gamma$ | Effect |
|----------|------------------|-------------------------------------|--------|
| Easy walking (negative) | High ($p_t \approx 1$) | Approaches 0 | Loss erased |
| Hard anomaly (confused) | Low ($p_t$ small) | Stays near 1 | Full focus |

#### Key Insight

> Focal Loss turns the model into a **specialized impact-detection sniper** — ignoring mundane activity to focus purely on complex fall physics.

**Why $\alpha = 0.85$?**  
A fall lasts ~10 frames in a 50-frame window. Missing a fall is far worse than a false alarm.

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ==========================================
# 0. CUSTOM FOCAL LOSS INJECTION
# ==========================================
class BinaryFocalLoss(nn.Module):
    """
    Focal Loss for Binary Classification.
    Addresses class imbalance and down-weights 'easy' examples.
    """
    def __init__(self, alpha=0.85, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha 
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = alpha_t * ((1 - p_t) ** self.gamma) * bce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

# ==========================================
# 1. THE FINAL HARDCODED BASELINE
# ==========================================
class FinalBaseline1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Optuna's Winning Specs
        filter_1 = 32
        filter_2 = 256
        kernel_size = 7
        self.dropout_rate = 0.2445
        
        pad = kernel_size // 2 
        
        self.conv1 = nn.Conv1d(10, filter_1, kernel_size=kernel_size, padding=pad)
        self.bn1 = nn.BatchNorm1d(filter_1)
        self.pool = nn.MaxPool1d(2) 
        self.conv2 = nn.Conv1d(filter_1, filter_2, kernel_size=kernel_size, padding=pad)
        self.bn2 = nn.BatchNorm1d(filter_2)
        
        self.dropout = nn.Dropout(self.dropout_rate)
        
        self.fc1 = nn.Linear(filter_2 * 25, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = torch.flatten(x, 1) 
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(1)

# ==========================================
# 2. LAUNCH FOCAL LOSS RUN
# ==========================================
print("=== LAUNCHING ULTIMATE BASELINE WITH FOCAL LOSS ===")

final_cnn = FinalBaseline1DCNN()

# THE MODIFICATION: Using the Focal Loss sniper rifle instead of standard BCE
criterion = BinaryFocalLoss(alpha=0.85, gamma=2.0)

# Optuna's exact winning learning rate
optimizer = optim.Adam(final_cnn.parameters(), lr=0.000915) 

trainer = ExperimentTrainer(
    exp_name="Exp_009_Ultimate_1DCNN_Focal",
    description="Final Tuned 1D CNN paired with custom Focal Loss (alpha=0.85, gamma=2.0).",
    model=final_cnn,
    criterion=criterion,
    optimizer=optimizer
)

# Train for 40 epochs
best_model, metrics = trainer.train(
    X_train_scaled, y_train, 
    X_val_scaled, y_val, 
    epochs=40, 
    batch_size=64 # Optuna's winning batch size
)

# ==========================================
# 3. THE BLIND TEST
# ==========================================
tester = ExperimentTester(
    exp_name="Exp_009_Ultimate_1DCNN_Focal", 
    model_architecture=FinalBaseline1DCNN()
)

preds, targets = tester.run_blind_test(
    X_test=X_test_scaled, 
    y_test=y_test, 
    test_name="Focal Loss CNN Blind Test", 
    extraction_method="Strict Overlap (50%)",
    batch_size=64
)

=== LAUNCHING ULTIMATE BASELINE WITH FOCAL LOSS ===

=== LAUNCHING EXPERIMENT: Exp_009_Ultimate_1DCNN_Focal ===
Device: cuda | Epochs: 40 | Batch: 64
Epoch [01/40] | Train Loss: 0.0104 | Val Loss: 0.0060 | Val F1: 0.7928 | Recall: 0.9660
Epoch [02/40] | Train Loss: 0.0075 | Val Loss: 0.0048 | Val F1: 0.8252 | Recall: 0.9684
Epoch [03/40] | Train Loss: 0.0063 | Val Loss: 0.0052 | Val F1: 0.8614 | Recall: 0.9539
Epoch [04/40] | Train Loss: 0.0061 | Val Loss: 0.0052 | Val F1: 0.8502 | Recall: 0.9745
Epoch [05/40] | Train Loss: 0.0059 | Val Loss: 0.0053 | Val F1: 0.8036 | Recall: 0.9757
Epoch [06/40] | Train Loss: 0.0052 | Val Loss: 0.0050 | Val F1: 0.8884 | Recall: 0.9563
Epoch [07/40] | Train Loss: 0.0050 | Val Loss: 0.0052 | Val F1: 0.8616 | Recall: 0.9709
Epoch [08/40] | Train Loss: 0.0047 | Val Loss: 0.0063 | Val F1: 0.8022 | Recall: 0.9818
Epoch [09/40] | Train Loss: 0.0046 | Val Loss: 0.0049 | Val F1: 0.8471 | Recall: 0.9782
Epoch [10/40] | Train Loss: 0.0042 | Val Loss: 0.0054 | Va